# Step 2 — Curated Operational Dataset and KPI Foundations

This notebook builds the star schema analytical model and derives KPI flags for the Power BI dashboard.

**Outputs:** `fact_flights.parquet`, `dim_airlines.csv`, `dim_airports.csv`, `dim_cancellation_codes.csv`

In [60]:
from __future__ import annotations

import os
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

print("Imports loaded.")

Imports loaded.


## 01 — Configuration and Paths

In [61]:
# Project root (Windows)
PROJECT_ROOT = Path(r"C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

RAW_FLIGHTS = RAW_DIR / "flights.csv"
RAW_AIRLINES = RAW_DIR / "airlines.csv"
RAW_AIRPORTS = RAW_DIR / "airports.csv"
RAW_CANCEL_CODES = RAW_DIR / "cancellation_codes.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_FLIGHTS exists: {RAW_FLIGHTS.exists()}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

PROJECT_ROOT: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio
RAW_FLIGHTS exists: True
PROCESSED_DIR: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed


In [62]:
RUN_ID = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d_%H%M%S")
print(f"RUN_ID: {RUN_ID}")

RUN_ID: 20260308_165228


## 02 — Utility Functions

In [63]:
def save_csv(df: pd.DataFrame, path: Path, *, index: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print(f"Saved CSV: {path} | rows={len(df):,} cols={df.shape[1]:,}")

def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    print(f"Saved Parquet: {path} | rows={len(df):,} cols={df.shape[1]:,}")

## 03 — Load Lookup Dimensions

In [64]:
airlines = pd.read_csv(RAW_AIRLINES)
airports = pd.read_csv(RAW_AIRPORTS)
cancel_codes = pd.read_csv(RAW_CANCEL_CODES)

print(f"airlines: {airlines.shape}")
print(f"airports: {airports.shape}")
print(f"cancel_codes: {cancel_codes.shape}")

display(airlines.head(3))
display(airports.head(3))
display(cancel_codes.head(3))

airlines: (14, 2)
airports: (322, 7)
cancel_codes: (4, 2)


,IATA_CODE,AIRLINE
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.


,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919


,CANCELLATION_REASON,CANCELLATION_DESCRIPTION
0,A,Airline/Carrier
1,B,Weather
2,C,National Air System


## 04 — Load Flights with Controlled Schema

In [65]:
FACT_USECOLS = [
    "YEAR", "MONTH", "DAY",
    "AIRLINE", "FLIGHT_NUMBER",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
    "SCHEDULED_DEPARTURE",

    # Optional drilldown
    "TAIL_NUMBER",

    # Status
    "CANCELLED", "CANCELLATION_REASON", "DIVERTED",

    # Delays
    "DEPARTURE_DELAY", "ARRIVAL_DELAY",

    # Delay components
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",

    # Ops measures
    "DISTANCE", "SCHEDULED_TIME", "ELAPSED_TIME", "AIR_TIME", "TAXI_OUT", "TAXI_IN",
]

In [66]:
DTYPE_MAP = {
    # Identity
    "YEAR": "Int16",
    "MONTH": "Int8",
    "DAY": "Int8",
    "FLIGHT_NUMBER": "Int32",

    # Flags
    "CANCELLED": "Int8",
    "DIVERTED": "Int8",

    # Codes
    "AIRLINE": "category",
    "ORIGIN_AIRPORT": "category",
    "DESTINATION_AIRPORT": "category",
    "CANCELLATION_REASON": "category",
    "TAIL_NUMBER": "category",

    # Delays
    "DEPARTURE_DELAY": "Int32",
    "ARRIVAL_DELAY": "Int32",

    # Delay components
    "AIR_SYSTEM_DELAY": "Int32",
    "SECURITY_DELAY": "Int32",
    "AIRLINE_DELAY": "Int32",
    "LATE_AIRCRAFT_DELAY": "Int32",
    "WEATHER_DELAY": "Int32",

    # Times
    "SCHEDULED_DEPARTURE": "Int32",
    "DEPARTURE_TIME": "Int32",
    "SCHEDULED_ARRIVAL": "Int32",
    "ARRIVAL_TIME": "Int32",
    "TAXI_OUT": "Int16",
    "SCHEDULED_TIME": "Int16",
    "ELAPSED_TIME": "Int16",
    "AIR_TIME": "Int16",
    "DISTANCE": "Int16",
    "WHEELS_ON": "Int32",
    "WHEELS_OFF": "Int32",
    "TAXI_IN": "Int16",
}

In [67]:
df = pd.read_csv(
    RAW_FLIGHTS,
    usecols=FACT_USECOLS,
    dtype=DTYPE_MAP
)

print(f"Loaded df: {df.shape}")
print(f"Memory (MB): {round(df.memory_usage(deep=True).sum() / 1024**2, 2)}")

Loaded df: (5819079, 25)
Memory (MB): 455.3


## 05 — Derive Time Slicing Columns

In [68]:
# Create flight_date
df["flight_date"] = pd.to_datetime(
    dict(year=df["YEAR"], month=df["MONTH"], day=df["DAY"]),
    errors="coerce"
).dt.date

# Derive scheduled departure hour from HHMM integer
sched = df["SCHEDULED_DEPARTURE"].astype("int32")
df["scheduled_departure_hour"] = (sched // 100).astype("int8")

# Day and month labels
dt = pd.to_datetime(df["flight_date"])
df["day_of_week"] = dt.dt.day_name()
df["month"] = dt.dt.month_name()

# Time band buckets
h = df["scheduled_departure_hour"]
df["time_band"] = pd.cut(
    h,
    bins=[-1, 5, 9, 15, 19, 23],
    labels=["EarlyAM", "AMPeak", "Midday", "PMPeak", "Evening"]
).astype("category")

df[[
    "flight_date",
    "SCHEDULED_DEPARTURE",
    "scheduled_departure_hour",
    "time_band",
    "day_of_week",
    "month"
]].head()

,flight_date,SCHEDULED_DEPARTURE,scheduled_departure_hour,time_band,day_of_week,month
0,2015-01-01,5,0,EarlyAM,Thursday,January
1,2015-01-01,10,0,EarlyAM,Thursday,January
2,2015-01-01,20,0,EarlyAM,Thursday,January
3,2015-01-01,20,0,EarlyAM,Thursday,January
4,2015-01-01,25,0,EarlyAM,Thursday,January


## 06 — Derive KPI Flags

In [69]:
# Eligibility: completed flights only (not cancelled, not diverted)
df["is_otp15_eligible"] = (
    (df["CANCELLED"] == 0) &
    (df["DIVERTED"] == 0)
).astype("Int8")

# On-Time Performance (OTP15): <= 15 mins arrival delay, only for eligible flights
df["is_on_time_otp15"] = (
    (df["is_otp15_eligible"] == 1) &
    (df["ARRIVAL_DELAY"] <= 15)
).astype("Int8")

# Delayed (OTP15): > 15 mins arrival delay, only for eligible flights
df["is_delayed_otp15"] = (
    (df["is_otp15_eligible"] == 1) &
    (df["ARRIVAL_DELAY"] > 15)
).astype("Int8")

# Severe Delay (SD60): >= 60 mins arrival delay, only for eligible flights
df["is_severe_delay_sd60"] = (
    (df["is_otp15_eligible"] == 1) &
    (df["ARRIVAL_DELAY"] >= 60)
).astype("Int8")

In [70]:
df[[
    "is_otp15_eligible",
    "is_on_time_otp15",
    "is_delayed_otp15",
    "is_severe_delay_sd60"
]].mean()

is_otp15_eligible       0.981944
is_on_time_otp15        0.806057
is_delayed_otp15        0.175887
is_severe_delay_sd60    0.055952
dtype: Float64

In [71]:
eligible = df["is_otp15_eligible"] == 1

otp15 = df.loc[eligible, "is_on_time_otp15"].mean()
severe_rate = df.loc[eligible, "is_severe_delay_sd60"].mean()

print(f"OTP15 % (eligible only): {round(otp15 * 100, 2)}%")
print(f"Severe Delay % (eligible only): {round(severe_rate * 100, 2)}%")

OTP15 % (eligible only): 82.09%
Severe Delay % (eligible only): 5.7%


## 07 — Derive Delay Buckets and Extreme Flag

In [72]:
# Delay bucket (only meaningful for eligible flights, but we classify all)

df["delay_bucket"] = pd.cut(
    df["ARRIVAL_DELAY"],
    bins=[-10_000, 15, 30, 59, 10_000],
    labels=["<=15 (On-time)", "16-30", "31-59", "60+"]
).astype("category")

# Extreme arrival delay flag (based on Step 1 discovery)
df["is_extreme_arrival_delay"] = (
    (df["ARRIVAL_DELAY"] < -60) |
    (df["ARRIVAL_DELAY"] > 1440)
).astype("Int8")

In [73]:
print("Delay bucket distribution:")
print(df["delay_bucket"].value_counts(normalize=True).round(4))
print(f"Extreme arrival delay rate: {df['is_extreme_arrival_delay'].mean():.6f}")

Delay bucket distribution:
delay_bucket
<=15 (On-time)    0.8209
16-30             0.0684
60+               0.0570
31-59             0.0537
Name: proportion, dtype: float64
Extreme arrival delay rate: 0.000074


## 08 — Derive Primary Delay Driver

In [74]:
delay_components = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]

comp_df = df[delay_components]

# Sum across components (all-NA -> 0, all-zero -> 0)
component_sum = comp_df.fillna(0).sum(axis=1)

# Max minutes 
df["primary_delay_driver_minutes"] = comp_df.fillna(0).max(axis=1)

# Safe idxmax: no error because we removed NA for comparison
df["primary_delay_driver"] = comp_df.fillna(0).idxmax(axis=1)

# Not delayed => not applicable (NULL)
not_delayed = df["is_delayed_otp15"] == 0
df.loc[not_delayed, "primary_delay_driver"] = pd.NA
df.loc[not_delayed, "primary_delay_driver_minutes"] = pd.NA

# Delayed but no attribution (all components 0 or NA) => Unattributed
delayed = df["is_delayed_otp15"] == 1
no_attribution = delayed & (component_sum == 0)

df.loc[no_attribution, "primary_delay_driver"] = "Unattributed"
df.loc[no_attribution, "primary_delay_driver_minutes"] = pd.NA

# Memory friendly
df["primary_delay_driver"] = df["primary_delay_driver"].astype("category")
print("Primary Delay Driver distribution (all):")
print(df["primary_delay_driver"].value_counts(dropna=False).head(10))
print("\nPrimary Delay Driver distribution (delayed only):")
print(df.loc[df["is_delayed_otp15"] == 1, "primary_delay_driver"].value_counts(normalize=True).round(3))

Primary Delay Driver distribution (all):
primary_delay_driver
NaN                    4795581
LATE_AIRCRAFT_DELAY     400438
AIRLINE_DELAY           299858
AIR_SYSTEM_DELAY        286330
WEATHER_DELAY            35059
SECURITY_DELAY            1813
Name: count, dtype: int64

Primary Delay Driver distribution (delayed only):
primary_delay_driver
LATE_AIRCRAFT_DELAY    0.391
AIRLINE_DELAY          0.293
AIR_SYSTEM_DELAY       0.280
WEATHER_DELAY          0.034
SECURITY_DELAY         0.002
Name: proportion, dtype: float64


## 09 — Derive Operational Disruption Flag

In [75]:
df["operational_disruption_flag"] = (
    (df["CANCELLED"] == 1) |
    (df["DIVERTED"] == 1) |
    ((df["is_otp15_eligible"] == 1) & (df["is_severe_delay_sd60"] == 1))
).astype("Int8")
print(f"Operational Disruption %: {round(df['operational_disruption_flag'].mean() * 100, 2)}%")

Operational Disruption %: 7.4%


## 10 — Build Fact Table

In [76]:
fact_cols = [
    # Keys / identity
    "flight_date",
    "AIRLINE", "FLIGHT_NUMBER",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
    "SCHEDULED_DEPARTURE",
    "TAIL_NUMBER",

    # Time slicers
    "scheduled_departure_hour", "time_band", "day_of_week", "month",

    # Status
    "CANCELLED", "CANCELLATION_REASON", "DIVERTED",

    # Measures
    "DISTANCE", "SCHEDULED_TIME", "ELAPSED_TIME", "AIR_TIME", "TAXI_OUT", "TAXI_IN",
    "DEPARTURE_DELAY", "ARRIVAL_DELAY",

    # Delay components
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",

    # KPI flags / features
    "is_otp15_eligible",
    "is_on_time_otp15",
    "is_delayed_otp15",
    "is_severe_delay_sd60",
    "delay_bucket",
    "is_extreme_arrival_delay",
    "primary_delay_driver",
    "primary_delay_driver_minutes",
    "operational_disruption_flag",
]

fact_flights = df[fact_cols].copy()

print(f"fact_flights shape: {fact_flights.shape}")
print(f"Memory (MB): {round(fact_flights.memory_usage(deep=True).sum() / 1024**2, 2)}")
fact_flights.head(3)

fact_flights shape: (5819079, 36)
Memory (MB): 917.16


,flight_date,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,TAIL_NUMBER,scheduled_departure_hour,time_band,day_of_week,month,CANCELLED,CANCELLATION_REASON,DIVERTED,DISTANCE,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,TAXI_OUT,TAXI_IN,DEPARTURE_DELAY,ARRIVAL_DELAY,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,is_otp15_eligible,is_on_time_otp15,is_delayed_otp15,is_severe_delay_sd60,delay_bucket,is_extreme_arrival_delay,primary_delay_driver,primary_delay_driver_minutes,operational_disruption_flag
0,2015-01-01,AS,98,ANC,SEA,5,N407AS,0,EarlyAM,Thursday,January,0,NaN,0,1448,205,194,169,21,4,-11,-22,<NA>,<NA>,<NA>,<NA>,<NA>,1,1,0,0,<=15 (On-time),0,NaN,<NA>,0
1,2015-01-01,AA,2336,LAX,PBI,10,N3KUAA,0,EarlyAM,Thursday,January,0,NaN,0,2330,280,279,263,12,4,-8,-9,<NA>,<NA>,<NA>,<NA>,<NA>,1,1,0,0,<=15 (On-time),0,NaN,<NA>,0
2,2015-01-01,US,840,SFO,CLT,20,N171US,0,EarlyAM,Thursday,January,0,NaN,0,2296,286,293,266,16,11,-2,5,<NA>,<NA>,<NA>,<NA>,<NA>,1,1,0,0,<=15 (On-time),0,NaN,<NA>,0


## 11 — Prepare Dimension Tables

In [77]:
dim_airlines = pd.read_csv(RAW_AIRLINES)
dim_airlines.columns = [c.strip().lower() for c in dim_airlines.columns]
dim_airlines = dim_airlines.rename(columns={
    "iata_code": "airline_code",
    "airline": "airline_name"
})

dim_airports = pd.read_csv(RAW_AIRPORTS)
dim_airports.columns = [c.strip().lower() for c in dim_airports.columns]
dim_airports = dim_airports.rename(columns={
    "iata_code": "airport_code",
    "airport": "airport_name"
})

dim_cancel = pd.read_csv(RAW_CANCEL_CODES)
dim_cancel.columns = [c.strip().lower() for c in dim_cancel.columns]
dim_cancel = dim_cancel.rename(columns={
    "cancellation_reason": "cancellation_code",
    "description": "cancellation_description"
})

print(f"dim_airlines: {dim_airlines.shape} | cols: {dim_airlines.columns.tolist()}")
print(f"dim_airports: {dim_airports.shape} | cols: {dim_airports.columns.tolist()}")
print(f"dim_cancel: {dim_cancel.shape} | cols: {dim_cancel.columns.tolist()}")

display(dim_airlines.head(3))
display(dim_airports.head(3))
display(dim_cancel.head(3))

dim_airlines: (14, 2) | cols: ['airline_code', 'airline_name']
dim_airports: (322, 7) | cols: ['airport_code', 'airport_name', 'city', 'state', 'country', 'latitude', 'longitude']
dim_cancel: (4, 2) | cols: ['cancellation_code', 'cancellation_description']


,airline_code,airline_name
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.


,airport_code,airport_name,city,state,country,latitude,longitude
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919


,cancellation_code,cancellation_description
0,A,Airline/Carrier
1,B,Weather
2,C,National Air System


## 12 — Referential Integrity Check

In [78]:
# 1) Airline codes in fact must exist in dim_airlines
missing_airlines = (
    set(fact_flights["AIRLINE"].dropna().unique())
    - set(dim_airlines["airline_code"].dropna().unique())
)

# 2) Airport codes in fact must exist in dim_airports (origin + dest)
missing_origin_airports = (
    set(fact_flights["ORIGIN_AIRPORT"].dropna().unique())
    - set(dim_airports["airport_code"].dropna().unique())
)
missing_dest_airports = (
    set(fact_flights["DESTINATION_AIRPORT"].dropna().unique())
    - set(dim_airports["airport_code"].dropna().unique())
)

# 3) Cancellation codes in fact must exist in dim_cancel (only where cancelled=1)
cancelled_codes = fact_flights.loc[fact_flights["CANCELLED"] == 1, "CANCELLATION_REASON"]
missing_cancel_codes = (
    set(cancelled_codes.dropna().unique())
    - set(dim_cancel["cancellation_code"].dropna().unique())
)

print(f"Missing airline codes: {len(missing_airlines)} {sorted(list(missing_airlines))[:10]}")
print(f"Missing origin airport codes: {len(missing_origin_airports)} {sorted(list(missing_origin_airports))[:10]}")
print(f"Missing dest airport codes: {len(missing_dest_airports)} {sorted(list(missing_dest_airports))[:10]}")
print(f"Missing cancellation codes: {len(missing_cancel_codes)} {sorted(list(missing_cancel_codes))[:10]}")

Missing airline codes: 0 []
Missing origin airport codes: 306 ['10135', '10136', '10140', '10141', '10146', '10154', '10155', '10157', '10158', '10165']
Missing dest airport codes: 307 ['10135', '10136', '10140', '10141', '10146', '10154', '10155', '10157', '10158', '10165']
Missing cancellation codes: 0 []


## 13 — Fix Missing Airport Codes

In [79]:
fact_flights["ORIGIN_AIRPORT"] = fact_flights["ORIGIN_AIRPORT"].astype(str)
fact_flights["DESTINATION_AIRPORT"] = fact_flights["DESTINATION_AIRPORT"].astype(str)
dim_airports["airport_code"] = dim_airports["airport_code"].astype(str)

In [80]:
missing_codes = sorted(list(
    (set(fact_flights["ORIGIN_AIRPORT"].unique()) | set(fact_flights["DESTINATION_AIRPORT"].unique()))
    - set(dim_airports["airport_code"].unique())
))

missing_airports_rows = pd.DataFrame({
    "airport_code": missing_codes,
    "airport_name": pd.NA,
    "city": pd.NA,
    "state": pd.NA,
    "country": pd.NA,
    "latitude": pd.NA,
    "longitude": pd.NA,
})

dim_airports_fixed = pd.concat([dim_airports, missing_airports_rows], ignore_index=True)
print(f"dim_airports_fixed: {dim_airports_fixed.shape}")
print(f"Added missing airport codes: {len(missing_codes)}")

dim_airports_fixed: (629, 7)
Added missing airport codes: 307


In [81]:
print("Airport DOT → IATA mapping")

airport_lookup_DOT = pd.read_csv(
    PROJECT_ROOT / "data/raw/L_AIRPORT_ID.csv",
    names=["DOT_ID", "AIRPORT_INFO"],
    skiprows=1
)

display(airport_lookup_DOT.head())

Airport DOT → IATA mapping


,DOT_ID,AIRPORT_INFO
0,10001,"Afognak Lake, AK: Afognak Lake Airport"
1,10003,"Granite Mountain, AK: Bear Creek Mining Strip"
2,10004,"Lik, AK: Lik Mining Camp"
3,10005,"Little Squaw, AK: Little Squaw Airport"
4,10006,"Kizhuyak, AK: Kizhuyak Bay"


In [82]:
print(airport_lookup_DOT.dtypes)

DOT_ID          int64
AIRPORT_INFO      str
dtype: object


In [83]:
print("Null DOT_ID:", airport_lookup_DOT["DOT_ID"].isna().sum())
print("Null AIRPORT_INFO:", airport_lookup_DOT["AIRPORT_INFO"].isna().sum())
print("Duplicate DOT_ID:", airport_lookup_DOT["DOT_ID"].duplicated().sum())
print("Row count:", len(airport_lookup_DOT))

Null DOT_ID: 0
Null AIRPORT_INFO: 0
Duplicate DOT_ID: 0
Row count: 6850


In [84]:
numeric_origin = fact_flights["ORIGIN_AIRPORT"].astype(str).str.isnumeric()
numeric_dest = fact_flights["DESTINATION_AIRPORT"].astype(str).str.isnumeric()

print("Numeric origin rows:", int(numeric_origin.sum()))
print("Numeric destination rows:", int(numeric_dest.sum()))

print("Unique numeric origin IDs:", fact_flights.loc[numeric_origin, "ORIGIN_AIRPORT"].nunique())
print("Unique numeric destination IDs:", fact_flights.loc[numeric_dest, "DESTINATION_AIRPORT"].nunique())

Numeric origin rows: 486165
Numeric destination rows: 486165
Unique numeric origin IDs: 306
Unique numeric destination IDs: 307


In [85]:
used_dot_ids = pd.Index(
    pd.concat([
        fact_flights.loc[numeric_origin, "ORIGIN_AIRPORT"].astype("int64"),
        fact_flights.loc[numeric_dest, "DESTINATION_AIRPORT"].astype("int64")
    ]).unique()
)

print("Used DOT IDs:", len(used_dot_ids))

Used DOT IDs: 307


In [86]:
airport_lookup_used = airport_lookup_DOT[
    airport_lookup_DOT["DOT_ID"].isin(used_dot_ids)
].copy()

print("Filtered DOT lookup rows:", len(airport_lookup_used))
display(airport_lookup_used.head(20))

Filtered DOT lookup rows: 307


,DOT_ID,AIRPORT_INFO
119,10135,"Allentown/Bethlehem/Easton, PA: Lehigh Valley International"
120,10136,"Abilene, TX: Abilene Regional"
124,10140,"Albuquerque, NM: Albuquerque International Sunport"
125,10141,"Aberdeen, SD: Aberdeen Regional"
130,10146,"Albany, GA: Southwest Georgia Regional"
138,10154,"Nantucket, MA: Nantucket Memorial"
139,10155,"Waco, TX: Waco Regional"
141,10157,"Arcata/Eureka, CA: California Redwood Coast Humboldt County"
142,10158,"Atlantic City, NJ: Atlantic City International"
149,10165,"Adak Island, AK: Adak"


In [87]:
missing_used_ids = used_dot_ids.difference(airport_lookup_used["DOT_ID"])

print("Used DOT IDs missing from lookup:", len(missing_used_ids))
print(missing_used_ids[:20])

Used DOT IDs missing from lookup: 0
Index([], dtype='int64')


In [88]:
display(airports.head())
print(airports.columns.tolist())

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447


['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE', 'LONGITUDE']


In [89]:
print("Split used DOT lookup into structured fields...")

split_parts = airport_lookup_used["AIRPORT_INFO"].str.split(":", n=1, expand=True)

airport_lookup_used["DOT_CITY_STATE"] = split_parts[0].str.strip()
airport_lookup_used["DOT_AIRPORT_NAME"] = split_parts[1].str.strip()

city_state_split = airport_lookup_used["DOT_CITY_STATE"].str.rsplit(",", n=1, expand=True)

airport_lookup_used["DOT_CITY"] = city_state_split[0].str.strip()
airport_lookup_used["DOT_STATE"] = city_state_split[1].str.strip()

display(
    airport_lookup_used[
        ["DOT_ID", "DOT_CITY", "DOT_STATE", "DOT_AIRPORT_NAME"]
    ].head(15)
)

Split used DOT lookup into structured fields...


,DOT_ID,DOT_CITY,DOT_STATE,DOT_AIRPORT_NAME
119,10135,Allentown/Bethlehem/Easton,PA,Lehigh Valley International
120,10136,Abilene,TX,Abilene Regional
124,10140,Albuquerque,NM,Albuquerque International Sunport
125,10141,Aberdeen,SD,Aberdeen Regional
130,10146,Albany,GA,Southwest Georgia Regional
138,10154,Nantucket,MA,Nantucket Memorial
139,10155,Waco,TX,Waco Regional
141,10157,Arcata/Eureka,CA,California Redwood Coast Humboldt County
142,10158,Atlantic City,NJ,Atlantic City International
149,10165,Adak Island,AK,Adak


In [90]:
airports.groupby(["AIRPORT", "STATE"]).size().sort_values(ascending=False).head(10)

AIRPORT                            STATE
Aberdeen Regional Airport          SD       1
Abilene Regional Airport           TX       1
Abraham Lincoln Capital Airport    IL       1
Adak Airport                       AK       1
Akron-Canton Regional Airport      OH       1
Albany International Airport       NY       1
Albert J. Ellis Airport            NC       1
Albuquerque International Sunport  NM       1
Alexandria International Airport   LA       1
Alpena County Regional Airport     MI       1
dtype: int64

In [91]:
airports_match = airports.copy()

airport_lookup_used["DOT_AIRPORT_NAME_NORM"] = (
    airport_lookup_used["DOT_AIRPORT_NAME"]
    .str.lower()
    .str.replace("airport", "", regex=False)
    .str.strip()
)

airports_match["AIRPORT_NORM"] = (
    airports_match["AIRPORT"]
    .str.lower()
    .str.replace("airport", "", regex=False)
    .str.strip()
)

In [92]:
def normalize_airport_name(name):
    if pd.isna(name):
        return None

    name = name.lower()

    # remove punctuation/brackets
    name = re.sub(r"[^\w\s]", " ", name)

    # remove low-information connector/suffix words
    name = re.sub(r"\bairport\b", " ", name)
    name = re.sub(r"\bat\b", " ", name)

    # collapse repeated whitespace
    name = re.sub(r"\s+", " ", name).strip()

    # sort words so ordering differences do not matter
    words = sorted(name.split())

    return " ".join(words)


# apply normalization
airport_lookup_used["AIRPORT_KEY"] = airport_lookup_used["DOT_AIRPORT_NAME"].apply(normalize_airport_name)

airports_match["AIRPORT_KEY"] = airports_match["AIRPORT"].apply(normalize_airport_name)

In [93]:
display(
    airport_lookup_used[
        airport_lookup_used["DOT_AIRPORT_NAME"].str.contains("augusta", case=False, na=False)
    ][["DOT_ID", "DOT_AIRPORT_NAME", "DOT_STATE", "AIRPORT_KEY"]]
)

display(
    airports_match[
        airports_match["AIRPORT"].str.contains("augusta", case=False, na=False)
    ][["IATA_CODE", "AIRPORT", "STATE", "AIRPORT_KEY"]]
)

,DOT_ID,DOT_AIRPORT_NAME,DOT_STATE,AIRPORT_KEY
189,10208,Augusta Regional at Bush Field,GA,augusta bush field regional


,IATA_CODE,AIRPORT,STATE,AIRPORT_KEY
12,AGS,Augusta Regional Airport (Bush Field),GA,augusta bush field regional


In [94]:
dot_to_iata_bridge = airport_lookup_used.merge(
    airports_match,
    left_on=["AIRPORT_KEY", "DOT_STATE"],
    right_on=["AIRPORT_KEY", "STATE"],
    how="left"
)

print("Bridge rows:", len(dot_to_iata_bridge))
print("Mapped:", int(dot_to_iata_bridge["IATA_CODE"].notna().sum()))
print("Unmapped:", int(dot_to_iata_bridge["IATA_CODE"].isna().sum()))

Bridge rows: 307
Mapped: 215
Unmapped: 92


In [95]:
dot_to_iata_bridge.groupby("DOT_ID")["IATA_CODE"].nunique().value_counts()

IATA_CODE
1    215
0     92
Name: count, dtype: int64

In [96]:
unmapped_dot = dot_to_iata_bridge[dot_to_iata_bridge["IATA_CODE"].isna()]

print("Unmapped DOT airports:", len(unmapped_dot))

display(
    unmapped_dot[
        ["DOT_ID", "DOT_AIRPORT_NAME", "DOT_CITY", "DOT_STATE"]
    ].head(20)
)

Unmapped DOT airports: 92


,DOT_ID,DOT_AIRPORT_NAME,DOT_CITY,DOT_STATE
7,10157,California Redwood Coast Humboldt County,Arcata/Eureka,CA
18,10372,Aspen Pitkin County Sardy Field,Aspen,CO
28,10577,Greater Binghamton/Edwin A. Link Field,Binghamton,NY
35,10685,Central Il Regional Airport at Bloomington,Bloomington/Normal,IL
37,10713,Boise Air Terminal,Boise,ID
38,10721,Logan International,Boston,MA
39,10728,Jack Brooks Regional,Beaumont/Port Arthur,TX
41,10732,Rafael Hernandez,Aguadilla,PR
46,10781,Baton Rouge Metropolitan/Ryan Field,Baton Rouge,LA
47,10785,Patrick Leahy Burlington International,Burlington,VT


In [97]:
unmatched_dot = dot_to_iata_bridge[dot_to_iata_bridge["IATA_CODE"].isna()].copy()

In [98]:
dot_to_iata = (
    dot_to_iata_bridge.dropna(subset=["IATA_CODE"])[["DOT_ID", "IATA_CODE"]]
    .drop_duplicates()
    .copy()
)

In [99]:
save_parquet(fact_flights, PROCESSED_DIR / "fact_flights_buck_up.parquet")

print("Saved pre-standardisation fact table snapshot.")

Saved Parquet: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\fact_flights_buck_up.parquet | rows=5,819,079 cols=36
Saved pre-standardisation fact table snapshot.


In [100]:
dot_to_iata = dot_to_iata.copy()
dot_to_iata["DOT_ID"] = dot_to_iata["DOT_ID"].astype(str)

fact_flights["ORIGIN_AIRPORT"] = fact_flights["ORIGIN_AIRPORT"].astype(str)
fact_flights["DESTINATION_AIRPORT"] = fact_flights["DESTINATION_AIRPORT"].astype(str)

In [101]:
# origin mapping
fact_flights = fact_flights.merge(
    dot_to_iata.rename(columns={"DOT_ID": "ORIGIN_AIRPORT", "IATA_CODE": "ORIGIN_IATA_MAPPED"}),
    on="ORIGIN_AIRPORT",
    how="left"
)

fact_flights["ORIGIN_AIRPORT"] = fact_flights["ORIGIN_IATA_MAPPED"].fillna(fact_flights["ORIGIN_AIRPORT"])
fact_flights = fact_flights.drop(columns=["ORIGIN_IATA_MAPPED"])

In [102]:
# destination mapping
fact_flights = fact_flights.merge(
    dot_to_iata.rename(columns={"DOT_ID": "DESTINATION_AIRPORT", "IATA_CODE": "DEST_IATA_MAPPED"}),
    on="DESTINATION_AIRPORT",
    how="left"
)

fact_flights["DESTINATION_AIRPORT"] = fact_flights["DEST_IATA_MAPPED"].fillna(fact_flights["DESTINATION_AIRPORT"])
fact_flights = fact_flights.drop(columns=["DEST_IATA_MAPPED"])

In [103]:
remaining_numeric_origin = fact_flights["ORIGIN_AIRPORT"].astype(str).str.isnumeric().sum()
remaining_numeric_dest = fact_flights["DESTINATION_AIRPORT"].astype(str).str.isnumeric().sum()

print("Remaining numeric ORIGIN_AIRPORT values:", int(remaining_numeric_origin))
print("Remaining numeric DESTINATION_AIRPORT values:", int(remaining_numeric_dest))

Remaining numeric ORIGIN_AIRPORT values: 141584
Remaining numeric DESTINATION_AIRPORT values: 141606


## 14 — Export Star Schema Outputs

In [104]:
save_parquet(fact_flights, PROCESSED_DIR / "fact_flights.parquet")
save_csv(dim_airlines, PROCESSED_DIR / "dim_airlines.csv")
save_csv(dim_airports, PROCESSED_DIR / "dim_airports.csv")
save_csv(dim_cancel, PROCESSED_DIR / "dim_cancellation_codes.csv")
save_csv(
    unmatched_dot[["DOT_ID", "DOT_AIRPORT_NAME", "DOT_STATE", "AIRPORT_KEY"]],
    PROCESSED_DIR / "unmatched_dot_airports.csv"
)

print("\nStep 2 complete. Star schema outputs saved.")

Saved Parquet: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\fact_flights.parquet | rows=5,819,079 cols=36
Saved CSV: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\dim_airlines.csv | rows=14 cols=2
Saved CSV: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\dim_airports.csv | rows=322 cols=7
Saved CSV: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\dim_cancellation_codes.csv | rows=4 cols=2
Saved CSV: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\unmatched_dot_airports.csv | rows=92 cols=4

Step 2 complete. Star schema outputs saved.


## Step 2 Summary

This notebook builds the curated analytical model for the Power BI dashboard.

**Generated outputs:**
- `data/processed/fact_flights.parquet` — main fact table with KPI flags
- `data/processed/dim_airlines.csv` — airline lookup dimension
- `data/processed/dim_airports.csv` — airport lookup dimension
- `data/processed/dim_cancellation_codes.csv` — cancellation reason lookup dimension

**KPI flags derived:**
- `is_otp15_eligible` — completed flights (not cancelled, not diverted)
- `is_on_time_otp15` — On-Time Performance (arrival delay ≤ 15 mins)
- `is_delayed_otp15` — delayed (arrival delay > 15 mins)
- `is_severe_delay_sd60` — Severe Delay (arrival delay ≥ 60 mins)
- `primary_delay_driver` — attribution of delay cause
- `operational_disruption_flag` — cancelled, diverted, or severely delayed